# Aivora AI - Kaggle GPU Training

Runs the real training pipeline (data prep -> train -> evaluate -> export) on
Kaggle's free GPU tier. Nothing here is simulated: every cell either does the
real thing or raises a `RuntimeError('STATUS = BLOCKED: ...')` naming exactly
what's missing, the same discipline used in `training/colab/FinLLM_GPU_Training.ipynb`.

## Before running

1. **Settings (right sidebar) -> Accelerator -> GPU T4 x2**
2. **Settings -> Internet -> On** (required for `git clone` and streaming
   datasets from Hugging Face)
3. Kaggle's free tier gives **30 GPU-hours/week**, and a single session can
   run up to ~9-12 hours before it's cut off. `small` (10M tokens, ~2,000
   steps) comfortably fits in one sitting. `financial_poc` (50M tokens,
   8,000 steps) may need to be resumed across 1-2 sessions - the trainer
   already checkpoints every `eval_interval` steps, so `--resume` picks up
   exactly where it left off; this notebook's training cell supports that
   out of the box.
4. To resume from a previous session's checkpoint: add it as a Kaggle
   Dataset (Notebook -> Add Data -> Upload) and set `RESUME_CHECKPOINT`
   below to its path under `/kaggle/input/...`.

## What this does NOT do

It does not fabricate a GPU, does not skip the leakage check, and does not
report a training result that didn't actually happen. If a cell's hard gate
raises, the notebook stops there - fix the named problem and re-run.


## 1. Environment

In [ ]:
import platform, sys, os
print("Python:", sys.version)
print("Platform:", platform.platform())
print("CWD:", os.getcwd())
print("Kaggle input mounted:", os.path.exists("/kaggle/input"), os.listdir("/kaggle/input") if os.path.exists("/kaggle/input") else [])


## 2. GPU / CUDA verification (hard gate)

Raises immediately if no GPU is attached - never claims GPU training happened without this passing.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "STATUS = BLOCKED: torch.cuda.is_available() is False. "
        "Go to Settings (right sidebar) -> Accelerator -> GPU T4 x2, "
        "save, and re-run this notebook from the top."
    )

GPU_NAME = torch.cuda.get_device_name(0)
GPU_COUNT = torch.cuda.device_count()
CC = torch.cuda.get_device_capability(0)
total_vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)

print("GPU AVAILABLE =", True)
print("GPU name:", GPU_NAME)
print("GPU count:", GPU_COUNT)
print("Compute capability:", CC)
print("Total VRAM (GPU 0): %.2f GB" % total_vram_gb)
print("torch version:", torch.__version__, "| CUDA build:", torch.version.cuda)


## 3. Repository transfer + integrity check

Clones the real, public repo. If this fails, Internet is probably off (Settings -> Internet -> On).

In [ ]:
import subprocess, os

REPO_URL = "https://github.com/Ankushk-aosc/Aivora-AI.git"
REPO_DIR = "/kaggle/working/Aivora-AI"

if not os.path.exists(REPO_DIR):
    result = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR],
                             capture_output=True, text=True)
    print(result.stdout)
    print(result.stderr)
    if result.returncode != 0:
        raise RuntimeError(
            "STATUS = BLOCKED: git clone failed (see stderr above). "
            "Most likely cause: Internet is off for this notebook "
            "(Settings -> Internet -> On), or the repo URL changed."
        )
else:
    print(f"{REPO_DIR} already present, skipping clone.")

os.chdir(REPO_DIR)
print("Now in:", os.getcwd())

required_paths = [
    "models/model.py", "training/trainer.py", "ai_platform/model_registry.py",
    "data_sources/prepare.py", "evaluation/evaluator.py", "configs/small.yaml",
    "configs/financial_poc.yaml", "inference/generator.py",
]
missing = [p for p in required_paths if not os.path.exists(p)]
if missing:
    raise RuntimeError(f"STATUS = BLOCKED: repo clone incomplete, missing {missing}")
print("Repo integrity check passed:", len(required_paths), "required paths present.")


## 4. Dependencies

Kaggle's base image already ships a CUDA-enabled PyTorch build tuned for the
attached GPU - reinstalling `torch` over it risks silently downgrading to a
CPU or mismatched-CUDA wheel. This only installs the *other* requirements,
and reuses `torch.cuda.is_available()` from Cell 2 to confirm nothing broke
it afterward.

In [ ]:
import subprocess, sys

with open("requirements.txt") as f:
    reqs = [line.strip() for line in f if line.strip() and not line.startswith("#")]

# torch is already provided by the Kaggle GPU image - skip it here.
reqs_to_install = [r for r in reqs if not r.lower().startswith("torch")]

result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q"] + reqs_to_install,
    capture_output=True, text=True,
)
print(result.stdout[-3000:])
print(result.stderr[-3000:])
if result.returncode != 0:
    raise RuntimeError("STATUS = BLOCKED: pip install failed, see output above.")

check = subprocess.run([sys.executable, "-m", "pip", "check"], capture_output=True, text=True)
print(check.stdout)
print(check.stderr)

# NOTE: deliberately do NOT importlib.reload(torch) here - torch's C
# extension init is not reload-safe (it re-registers native TORCH_LIBRARY
# namespaces with the dispatcher, which crashes on a second registration).
# Since torch itself was excluded from the install above, the already-
# imported torch module from Cell 2 is still valid and doesn't need reloading.
if not torch.cuda.is_available():
    raise RuntimeError(
        "STATUS = BLOCKED: torch.cuda.is_available() became False after "
        "installing requirements.txt - something in the dependency list "
        "pulled in a CPU-only torch build. Check requirements.txt for a "
        "torch pin and remove it."
    )
print("Dependencies installed; CUDA still available after install.")


## 5. Dataset preparation

Prepares real data for every bucket in the chosen preset's `dataset_mix`,
streamed live from Hugging Face with a hard token budget per Part 9/37 of
this project - nothing is fully downloaded, and every dataset used is the
one already verified in `data_sources/dataset_registry.py` (license +
schema checked against a live sample, not assumed).

In [ ]:
PRESET = "financial_poc"  # "small" (10M tokens, ~2,000 steps) or "financial_poc" (50M tokens, ~8,000 steps)

import yaml
from data_sources.dataset_registry import list_entries
from data_sources.dataset_mixer import BUCKET_TO_CATEGORY, validate_mix
from data_sources.prepare import prepare_dataset

with open(f"configs/{PRESET}.yaml") as f:
    preset_cfg = yaml.safe_load(f)

mix = preset_cfg["dataset_mix"]
validate_mix(mix)
total_budget = int(preset_cfg["train_tokens"]) + int(preset_cfg["validation_tokens"])
print(f"Preset '{PRESET}': total token budget {total_budget:,} across {len(mix)} buckets")

summary = []
for bucket, weight in mix.items():
    category = BUCKET_TO_CATEGORY[bucket]
    entries = list_entries(category=category, verified_only=True)
    if not entries:
        print(f"  [SKIP] bucket '{bucket}' (category '{category}') has no VERIFIED datasets registered.")
        continue
    bucket_budget = int(total_budget * weight)
    per_dataset_budget = max(bucket_budget // len(entries), 50_000)
    for entry in entries:
        print(f"  Preparing '{entry.name}' (bucket '{bucket}', budget {per_dataset_budget:,} tokens)...")
        result = prepare_dataset(entry.name, max_tokens=per_dataset_budget)
        summary.append((entry.name, result["train_tokens_used"], result["validation_tokens_used"]))

print()
print("Prepared datasets (real, measured token counts):")
for name, train_tok, val_tok in summary:
    print(f"  {name}: {train_tok:,} train / {val_tok:,} validation tokens")


## 6. Leakage check (hard gate)

Confirms no evaluation question appears verbatim in the training shards just prepared.

In [ ]:
from evaluation import check_leakage

leak_report = check_leakage()
print(leak_report)
if not leak_report.get("clean", False):
    raise RuntimeError(f"STATUS = BLOCKED: leakage detected - {leak_report}")
print("Leakage check passed: no evaluation text found in training shards.")


## 7. Tokenizer round-trip check

In [ ]:
from data_sources.tokenizer import get_encoding

enc = get_encoding()
sample = "EBITDA margin is calculated as EBITDA divided by revenue, times 100."
ids = enc.encode_ordinary(sample)
decoded = enc.decode(ids)
assert decoded == sample, f"Tokenizer round-trip failed: {decoded!r} != {sample!r}"
print(f"Tokenizer round-trip OK ({len(ids)} tokens for {len(sample)} chars).")


## 8. Model configuration

In [ ]:
from models import DeepSeekConfig, DeepSeekV3

config = DeepSeekConfig.default()
model_preview = DeepSeekV3(config)
param_count = sum(p.numel() for p in model_preview.parameters())
print(f"Model: {param_count:,} parameters")
print(config.to_dict())
del model_preview


## 9. Checkpoint discovery + compatibility check (optional resume)

If you attached a previous checkpoint as a Kaggle Dataset input, point
`RESUME_CHECKPOINT` at it. Real tensor-shape compatibility is checked
against a freshly-built reference model before trusting it - a checkpoint
that doesn't match this repo's current architecture is rejected with a
clear reason rather than silently corrupting the run.

In [ ]:
RESUME_CHECKPOINT = "/kaggle/input/notebooks/aoscjkjhh/aivora-ai-financial-poc-training/Aivora-AI/checkpoints/base/checkpoint_8000.pt"

# Kaggle mounts a kernel_source's output under /kaggle/input/<kernel-slug>/,
# preserving its original directory structure - the exact prefix (whether
# it keeps the "Aivora-AI/" segment or not) isn't 100% certain without
# having run it, so search for the real file rather than gamble a multi-
# hour training run on a guessed path. Falls back to the guess above only
# if nothing is found by the search, and raises BLOCKED (not silently
# skips resume) if truly nothing is there.
import glob as _glob
if RESUME_CHECKPOINT and not os.path.exists(RESUME_CHECKPOINT):
    candidates = _glob.glob("/kaggle/input/**/checkpoint_8000.pt", recursive=True)
    if candidates:
        print(f"Guessed resume path not found; located checkpoint via search instead: {candidates[0]}")
        RESUME_CHECKPOINT = candidates[0]

import torch
from models import DeepSeekConfig, DeepSeekV3

if RESUME_CHECKPOINT:
    if not os.path.exists(RESUME_CHECKPOINT):
        raise RuntimeError(f"STATUS = BLOCKED: {RESUME_CHECKPOINT} not found.")

    ref_model = DeepSeekV3(DeepSeekConfig.default())
    ref_state = ref_model.state_dict()

    ckpt = torch.load(RESUME_CHECKPOINT, map_location="cpu")
    ckpt_state = ckpt["model_state_dict"] if "model_state_dict" in ckpt else ckpt

    mismatches = []
    for key, ref_tensor in ref_state.items():
        if key not in ckpt_state:
            mismatches.append(f"missing key: {key}")
        elif ckpt_state[key].shape != ref_tensor.shape:
            mismatches.append(f"shape mismatch on {key}: checkpoint has "
                               f"{ckpt_state[key].shape}, current model expects {ref_tensor.shape}")
    if mismatches:
        raise RuntimeError(
            "STATUS = BLOCKED: checkpoint is not architecture-compatible with "
            f"the current model config: {mismatches[:5]}"
        )
    print(f"Checkpoint {RESUME_CHECKPOINT} is architecture-compatible. Will resume from it.")
    del ref_model
else:
    print("No RESUME_CHECKPOINT set - training will start from scratch.")

## 10. Training

Runs the real training loop (`training.trainer.train_model`), auto-detecting
the GPU. Catches genuine CUDA OOM and halves batch size, then sequence
length, retrying for real rather than assuming a config that was never
tested on this GPU. Checkpoints save every `eval_interval` steps (see the
preset yaml) - if this session is cut off by Kaggle's time limit, re-run
this notebook with `RESUME_CHECKPOINT` set to the last saved checkpoint
under `checkpoints/base/`.

In [ ]:
import yaml
from training.trainer import train_model

CFG_PATH = f"configs/{PRESET}.yaml"


def attempt_training(preset_name, resume, max_retries=4):
    import torch
    attempt = 0
    while attempt < max_retries:
        try:
            return train_model(preset_name=preset_name, resume=resume)
        except torch.cuda.OutOfMemoryError as e:
            attempt += 1
            print(f"CUDA OOM on attempt {attempt}/{max_retries}: {e}")
            torch.cuda.empty_cache()
            if attempt >= max_retries:
                raise RuntimeError(
                    f"STATUS = BLOCKED: repeated CUDA OOM after {max_retries} attempts "
                    f"on preset '{preset_name}'. batch_size/seq_len are already reduced "
                    f"as far as this notebook will go automatically - see "
                    f"configs/{preset_name}.yaml to reduce further by hand."
                )
            # Actually shrink the config on disk before retrying - train_model()
            # re-reads configs/{preset}.yaml fresh via load_preset() every call,
            # so mutating a local Python variable here would do nothing; this
            # was a real bug in an earlier version of this cell (it retried the
            # IDENTICAL config three times and failed identically every time).
            with open(CFG_PATH) as f:
                cfg = yaml.safe_load(f)
            if cfg["batch_size"] > 2:
                cfg["batch_size"] = max(2, cfg["batch_size"] // 2)
                print(f"Reducing batch_size to {cfg['batch_size']} and retrying.")
            elif cfg.get("seq_len") and cfg["seq_len"] > 128:
                cfg["seq_len"] = max(128, cfg["seq_len"] // 2)
                print(f"Reducing seq_len to {cfg['seq_len']} and retrying.")
            else:
                raise RuntimeError(
                    "STATUS = BLOCKED: batch_size/seq_len are already at the "
                    "minimum this notebook will try automatically."
                )
            with open(CFG_PATH, "w") as f:
                yaml.safe_dump(cfg, f)


model, config, ckpt_path = attempt_training(PRESET, RESUME_CHECKPOINT)
print(f"Training complete. Final checkpoint: {ckpt_path}")

## 11. Evaluation

In [ ]:
from evaluation import evaluate_model, print_report

eval_results = evaluate_model(model, device="cuda", max_new_tokens=48, verbose=True)
print_report(eval_results)


## 12. Inference test (the 3 required prompts)

In [ ]:
from inference import load_model_for_inference, generate_text

test_prompts = [
    "What is EBITDA?",
    "Calculate EBITDA margin for revenue 500 and EBITDA 100.",
    "What is working capital?",
]
for prompt in test_prompts:
    output = generate_text(ckpt_path, prompt, max_tokens=60, temperature=0.7, top_k=40, device="cuda")
    print(f"PROMPT: {prompt}")
    print(f"OUTPUT: {output}")
    print("-" * 60)


## 13. Checkpoint export + hash verification

In [ ]:
import json
from ai_platform.model_registry import register_checkpoint, verify_integrity

entry = register_checkpoint(ckpt_path, stage="base", set_active=True)
print("Registered checkpoint:", entry)

verification = verify_integrity(entry["version"])
print("Integrity verification:", verification)
if not verification["valid"]:
    raise RuntimeError(f"STATUS = BLOCKED: checkpoint integrity verification failed: {verification}")

manifest = {
    "preset": PRESET,
    "checkpoint_path": ckpt_path,
    "gpu": GPU_NAME,
    "registry_entry": entry,
}
with open("export_manifest.json", "w") as f:
    json.dump(manifest, f, indent=2, default=str)
print("Wrote export_manifest.json")


## 14. Bringing the checkpoint back to your local project

1. In the Kaggle sidebar, open **Output** and download the checkpoint
   `.pt` + `.json` pair from `checkpoints/base/` (and `export_manifest.json`).
2. Place them in your local repo under `checkpoints/base/`.
3. Register it locally:
   ```bash
   python -c "from ai_platform.model_registry import register_checkpoint; print(register_checkpoint('checkpoints/base/<checkpoint_name>.pt', stage='base'))"
   ```
4. Restart the backend pointed at the new checkpoint and re-run
   `python ai_platform/acceptance_test.py` against it.
5. Only after that passes should this move from "trained on Kaggle" to
   `LLM TRAINING STATUS = TESTED` in the project's registry - this notebook
   proves the GPU run happened; the acceptance suite proves it's wired into
   the real serving stack correctly.
